In [1]:
#r "nuget: ScottPlot, 5.0.39"


Installed Packages ScottPlot, 5.0.39

Loading extensions from `C:\Users\user\.nuget\packages\skiasharp\2.88.8\interactive-extensions\dotnet\SkiaSharp.DotNet.Interactive.dll`

In [2]:
#r "task17/bin/Debug/net10.0/task17.dll"
#r "task17Tests/bin/Debug/net10.0/task17Tests.dll"
using System;
using System.Diagnostics;
using System.Linq;
using System.Threading;
using System.Collections.Generic;
using ScottPlot;
using task17;

public class HeavyCommand : ILongCommand
{
    private int _rem;
    private readonly string _name;
    private readonly List<string>? _trace;

    public bool IsCompleted => _rem <= 0;

    public HeavyCommand(int n, string name, List<string>? trace = null)
    {
        _rem = n;
        _name = name;
        _trace = trace;
    }
    
    public void Execute()
    {
        if(IsCompleted)
            return;

        Thread.SpinWait(100000);

        int step = 10 - _rem + 1;
        _trace?.Add($"{_name}{step}");

        _rem--;
    }
}

Console.WriteLine("Один поток / Round Robin\n");

var singleTrace = new List<string>();
var rrTrace = new List<string>();

var counts = new int[] { 1, 2, 4, 8, 16 };
var singleTimes = new double[counts.Length];
var rrTimes = new double[counts.Length];

for (int i = 0; i < counts.Length; i++)
{
    int n = counts[i];
    int execPerCmd = 10;

    double singleTotal = 0;

    singleTrace.Clear();
    rrTrace.Clear();

    for (int run = 0; run < 3; run++)
    {
        var commands = new List<HeavyCommand>();

        for (int j = 0; j < n; j++)
            commands.Add(new HeavyCommand(execPerCmd, $"C{j}", n == 4 && run == 0 ? singleTrace : null));
        
        var sw = Stopwatch.StartNew();

        foreach (var cmd in commands)
        {
            while (!cmd.IsCompleted)
                cmd.Execute();
        }

        sw.Stop();

        singleTotal += sw.Elapsed.TotalMilliseconds;
    }

    singleTimes[i] = singleTotal / 3;


    double rrTotal = 0;

    for (int run = 0; run < 3; run++)
    {
        var s = new RoundRobinScheduler();
        var st = new ServerThread(s);
        var sw = Stopwatch.StartNew();

        for (int j = 0; j < n; j++)
            st.Add(new HeavyCommand(execPerCmd, $"C{j}", n == 4 && run == 0 ? rrTrace : null));
        st.Start();

        st.SoftStop();
        st.Join();

        sw.Stop();

        rrTotal += sw.Elapsed.TotalMilliseconds;
    }

    rrTimes[i] = rrTotal / 3;


    Console.WriteLine($"Команд: {n,2} | Один поток: {singleTimes[i],8:F1} мс | Round Robin: {rrTimes[i],8:F1} мс");

    if (n == 4)
    {
        Console.WriteLine();
        Console.WriteLine("Один поток:");
        Console.WriteLine(string.Join(" ", singleTrace));

        Console.WriteLine();
        Console.WriteLine("Round Robin:");
        Console.WriteLine(string.Join(" ", rrTrace));
        Console.WriteLine();
    }
}

var plot = new Plot();
plot.Title("Время выполнения: Один поток vs Round Robin");
plot.XLabel("Количество команд");
plot.YLabel("Время (мс)");

var scatterSingle = plot.Add.Scatter(counts.Select(x => (double)x).ToArray(), singleTimes);
scatterSingle.LegendText = "Один поток";

var scatterRR = plot.Add.Scatter(counts.Select(x => (double)x).ToArray(), rrTimes);
scatterRR.LegendText = "Round Robin";

plot.ShowLegend();

var fn = "plot.png";
plot.SavePng(fn, 800, 600);
display(HTML($"<img src='{fn}?t={DateTime.Now.Ticks}' width='700'/>"));

Один поток / Round Robin

Команд:  1 | Один поток:     42.4 мс | Round Robin:     39.5 мс
Команд:  2 | Один поток:     88.7 мс | Round Robin:     84.2 мс
Команд:  4 | Один поток:    168.4 мс | Round Robin:    203.5 мс

Один поток:
C01 C02 C03 C04 C05 C06 C07 C08 C09 C010 C11 C12 C13 C14 C15 C16 C17 C18 C19 C110 C21 C22 C23 C24 C25 C26 C27 C28 C29 C210 C31 C32 C33 C34 C35 C36 C37 C38 C39 C310

Round Robin:
C01 C11 C21 C31 C02 C12 C22 C32 C03 C13 C23 C33 C04 C14 C24 C34 C05 C15 C25 C35 C06 C16 C26 C36 C07 C17 C27 C37 C08 C18 C28 C38 C09 C19 C29 C39 C010 C110 C210 C310

Команд:  8 | Один поток:    335.1 мс | Round Robin:    323.2 мс
Команд: 16 | Один поток:    621.6 мс | Round Robin:    641.1 мс


In [3]:
#r "task17/bin/Debug/net10.0/task17.dll"

using task17;
using System.Threading;

var server = new ServerThread();

for (int i = 1; i <= 5; i++)
{
    server.Add(new TestCommand(i));
}

server.Start();

Thread.Sleep(1000);

server.Add(new HardStopCommand(server));

server.Join();

Поток 1 вызов 1
Поток 2 вызов 1
Поток 3 вызов 1
Поток 4 вызов 1
Поток 5 вызов 1
Поток 1 вызов 2
Поток 2 вызов 2
Поток 3 вызов 2
Поток 4 вызов 2
Поток 5 вызов 2
Поток 1 вызов 3
Поток 2 вызов 3
Поток 3 вызов 3
Поток 4 вызов 3
Поток 5 вызов 3
